In [11]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class TestCase:
    inputs: tuple
    expected: object

@dataclass
class Problem:
    id: int
    title: str
    difficulty: str
    category: str
    prompt: str
    function_name: str
    test_cases: list[TestCase]

@dataclass
class EvaluationResult:
    problem_id: int
    model: str
    prompt_strategy: str
    passed: bool
    tests_passed: int
    tests_total: int
    runtime_ms: Optional[float]
    error_type: Optional[str]
    error_message: Optional[str]

In [12]:
import json
from pathlib import Path


DATA_PATH = Path("../data/problems.json")


with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_problems = json.load(f)


raw_problems

[{'id': 1,
  'title': 'Two Sum',
  'difficulty': 'Easy',
  'category': 'Array',
  'prompt': 'Given an array of integers nums and an integer target, return indices of the two numbers such that they add up to target. You may assume that each input has exactly one solution.',
  'function_name': 'twoSum',
  'test_cases': [{'input': [[2, 7, 11, 15], 9], 'expected': [0, 1]},
   {'input': [[3, 2, 4], 6], 'expected': [1, 2]},
   {'input': [[3, 3], 6], 'expected': [0, 1]}]}]

In [13]:
problems = []

for p in raw_problems:
    test_cases = [
        TestCase(
            inputs=tuple(tc["input"]),
            expected=tc["expected"]
        )
        for tc in p["test_cases"]
    ]

    problems.append(
        Problem(
            id=p["id"],
            title=p["title"],
            difficulty=p["difficulty"],
            category=p["category"],
            prompt=p["prompt"],
            function_name=p["function_name"],
            test_cases=test_cases
        )
    )

problems

[Problem(id=1, title='Two Sum', difficulty='Easy', category='Array', prompt='Given an array of integers nums and an integer target, return indices of the two numbers such that they add up to target. You may assume that each input has exactly one solution.', function_name='twoSum', test_cases=[TestCase(inputs=([2, 7, 11, 15], 9), expected=[0, 1]), TestCase(inputs=([3, 2, 4], 6), expected=[1, 2]), TestCase(inputs=([3, 3], 6), expected=[0, 1])])]

In [14]:
def run_tests(solution_function, test_cases):
    results = []

    for test in test_cases:
        try:
            actual = solution_function(*test.inputs)

            passed = actual == test.expected

            results.append({
                "passed": passed,
                "expected": test.expected,
                "actual": actual,
                "error": None
            })

        except Exception as e:
            results.append({
                "passed": False,
                "expected": test.expected,
                "actual": None,
                "error": str(e)
            })

    return results

In [15]:
def two_sum(nums, target):
    lookup = {}

    for i, num in enumerate(nums):
        complement = target - num

        if complement in lookup:
            return [lookup[complement], i]

        lookup[num] = i

In [16]:
problem = problems[0]

test_results = run_tests(
    two_sum,
    problem.test_cases
)

test_results

[{'passed': True, 'expected': [0, 1], 'actual': [0, 1], 'error': None},
 {'passed': True, 'expected': [1, 2], 'actual': [1, 2], 'error': None},
 {'passed': True, 'expected': [0, 1], 'actual': [0, 1], 'error': None}]

In [17]:
def calculate_metrics(test_results):
    total = len(test_results)
    passed = sum(result["passed"] for result in test_results)

    return {
        "tests_total": total,
        "tests_passed": passed,
        "test_accuracy": passed / total if total else 0,
        "problem_solved": passed == total
    }

In [18]:
metrics = calculate_metrics(test_results)

metrics

{'tests_total': 3,
 'tests_passed': 3,
 'test_accuracy': 1.0,
 'problem_solved': True}

In [20]:
def print_evaluation(problem, metrics):
    print(f"Problem: {problem.title}")
    print(f"Difficulty: {problem.difficulty}")
    print(f"Category: {problem.category}")
    print()
    print(f"Tests passed: {metrics['tests_passed']}/{metrics['tests_total']}")
    print(f"Test accuracy: {metrics['test_accuracy']:.1%}")
    print(f"Problem solved: {metrics['problem_solved']}")

print_evaluation(problem, metrics)

Problem: Two Sum
Difficulty: Easy
Category: Array

Tests passed: 3/3
Test accuracy: 100.0%
Problem solved: True
